In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.6 The Classical Ideal Gas: Phase-Space Volume, Chemical Potential, and the Fundamental Relation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.6",
    title="The Classical Ideal Gas: Phase-Space Volume, Chemical Potential, and the Fundamental Relation",
    blurb="The microcanonical ensemble for a continuous system. We compute the "
    "entropy of an ideal gas from the genuine volume of its energy shell in "
    "6N-dimensional phase space, then read off the equation of state, the "
    "chemical potential, and the master relation dU = TdS − PdV + μdN. The "
    "velocity distribution falls out as the shadow of a high-dimensional "
    "sphere, and the one thing classical counting gets wrong points the way "
    "to the quantum volume.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

[§5.4](microstates-entropy-temperature.ipynb) built the microcanonical ensemble for a *discrete* system — spins we could count
one configuration at a time — and [§5.5](ergodicity.ipynb) reassured us the ensemble averages it computes are the
ones a real system measures. Now we pay the price of going *continuous*, and the price is the
whole point. A gas has no microstates to count: its phase space is a continuum, so its
multiplicity is a **volume** in the $6N$-dimensional space of all positions and momenta. Computing
that volume honestly — the volume of a ball in three thousand dimensions, regulated by a $\Gamma$
function — is real work, and this notebook does it rather than wave it away. The reward is
everything: from that one integral, *all* of equilibrium thermodynamics follows by
differentiation. The ideal gas law, the energy, the chemical potential, and the master equation
$dU=T\,dS-P\,dV+\mu\,dN$ are each a derivative of a single entropy.

The classical route is harder than the binomial coefficient of [§5.4](microstates-entropy-temperature.ipynb), and that is correct: we earn the
equation of state by doing the integral, not by assuming the answer. The arsenal carries us
through. The phase-space volume overflows any float, so we live in log space with
`scipy.special.gammaln` (the lesson of [§5.3](large-n-limit.ipynb) and [§0.1](../00-foundations/floating-point.ipynb), now in $3N$ dimensions). The worry that the
answer depends on an arbitrary shell thickness dissolves through **concentration of measure** —
the astonishing high-dimensional fact that almost all of a ball sits in a whisker-thin skin near
its surface, a geometric cousin of the large-$N$ sharpening of [§5.3](large-n-limit.ipynb). And the **Maxwell–Boltzmann**
velocity law turns out to be the *shadow* of the energy sphere projected onto one axis: a Gaussian
by the same central-limit mechanism of [§5.3](large-n-limit.ipynb), revealing the geometric origin of the Boltzmann factor
$e^{-mv^2/2kT}$.

We will (1) build the phase-space volume, (2) prove the shell and the ball carry the same
entropy, (3,4) extract $E=\tfrac32NkT$ and $PV=NkT$ from first principles, (5) compute the
chemical potential, (6) assemble the fundamental relation, (7) derive Maxwell's 1860 law as a
sphere projection, (8) meet the Gibbs paradox — the one place classical counting genuinely fails,
an honest cliffhanger to Volume VII — and (9) extend equipartition. We work in units
$m=h=k_B=1$ throughout, stated wherever it matters.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the $V^N\times$($3N$-ball) volume; the outer-shell concentration and shell–ball agreement;
> $E=\tfrac32NkT$ and $PV=NkT$ from $\partial S/\partial E$ and $\partial S/\partial V$; the
> chemical potential $\mu=-T\,\partial S/\partial N$; the fundamental relation as a numerical
> identity; a velocity component Gaussian with variance $kT/m$; the Gibbs factor restoring
> extensivity; and classical equipartition. A ✓ is strong evidence; a ✗ is a prompt to *locate
> the discrepancy*.
>
> **Scope.** The classical microcanonical ideal gas and the fundamental relation. Thermodynamic
> potentials, Legendre transforms, and Maxwell relations are [§5.7](potentials-legendre-maxwell.ipynb); the grand canonical ensemble
> (and $\mu$'s starring role) later in the volume; the *absolute* entropy (Sackur–Tetrode) and the
> origin of the $1/N!$ are Volume VII. See Schroeder, *An Introduction to Thermal Physics*; Kardar,
> *Statistical Physics of Particles*; Maxwell (1860); and [§5.4](microstates-entropy-temperature.ipynb), [§5.3](large-n-limit.ipynb), [§0.1](../00-foundations/floating-point.ipynb).

## Theory in brief

### The phase-space volume of a classical system

For a classical system the microcanonical multiplicity is a **phase-space volume**, not a count.
For $N$ free particles in a box of volume $V$ with energy $\le E$, the accessible region is $V^N$
(positions) times the volume of the momentum ball of radius $\sqrt{2mE}$ in $3N$ dimensions. The
$n$-ball volume is $V_n(R)=\pi^{n/2}R^n/\Gamma(n/2+1)$, so

```{math}
:label: eq-phase-volume
\Sigma(E,V,N)=\frac{1}{N!\,h^{3N}}\,V^N\,\frac{\pi^{3N/2}}{\Gamma(\tfrac{3N}{2}+1)}\,(2mE)^{3N/2}.
```

(the result of which is derived in every standard classical statistical mechanics course; as it
is a routine pencil-and-paper exercise, and emphatically *not* per se of computational interest,
we do not reproduce the derivation here — Schroeder, *An Introduction to Thermal Physics*,
§2.5, carries it out in full). The $1/h^{3N}$ makes $\Sigma$ dimensionless ($h$ sets
the size of a phase-space cell — its value is fixed by quantum mechanics, but it only shifts the
entropy by an additive constant), and the
$1/N!$ is the **Gibbs indistinguishability factor** (also quantum in origin; its role is Exercise
8). We compute $\ln\Sigma$ with `scipy.special.gammaln` because $\Sigma$ overflows ([§5.3](large-n-limit.ipynb)/[§0.1](../00-foundations/floating-point.ipynb), now
in $3N$ dimensions).

### Shell versus ball, and concentration of measure

The microcanonical ensemble lives on the energy **shell** ($E$ in $[E,E+\Delta E]$), of volume
$g(E)\Delta E$ with the density of states $g=\partial\Sigma/\partial E$. Does the entropy depend
on the arbitrary thickness $\Delta E$, or on shell-versus-ball? At large $N$, no — because of
**concentration of measure**: in high dimensions almost all of a ball's volume lies in a thin
outer shell,

```{math}
:label: eq-concentration
\frac{\text{outer-}\varepsilon\text{ shell}}{\text{ball}}=1-(1-\varepsilon)^n\xrightarrow{n\to\infty}1 ,
```

so $\ln\Sigma\approx\ln(g\,\Delta E)$ to $O(\ln N)$, and the entropy per particle is the same
whichever we use. This is the geometric cousin of the large-$N$ sharpening of [§5.3](large-n-limit.ipynb).

### Entropy, the equations of state, and the chemical potential

With $S=k\ln\Omega$ ([§5.4](microstates-entropy-temperature.ipynb)) the derivatives give everything,

```{math}
:label: eq-eos
\frac1T=\frac{\partial S}{\partial E}\;\Rightarrow\;E=\tfrac32 NkT ,\qquad
\frac PT=\frac{\partial S}{\partial V}\;\Rightarrow\;PV=NkT ,
```

the caloric and thermal equations of state — the ideal gas law *derived*, not assumed. The third
derivative is the **chemical potential**, the energy cost of adding a particle,

```{math}
:label: eq-mu
\mu=-T\,\frac{\partial S}{\partial N} ,
```

negative for a dilute classical gas, and the quantity that governs particle exchange (central to
the grand canonical ensemble). Assembling all three derivatives gives the **fundamental
thermodynamic relation**,

```{math}
:label: eq-fundamental
dU=T\,dS-P\,dV+\mu\,dN ,
```

the master equation of equilibrium thermodynamics — it packages the first and second laws and
defines $T,P,\mu$ as entropy's conjugate slopes. [§5.7](potentials-legendre-maxwell.ipynb) builds the potentials on it.

### Maxwell–Boltzmann as the shadow of the sphere

With total kinetic energy fixed, the $3N$ velocity components lie *uniformly* on a sphere of
radius $\sqrt{2E/m}$ in $3N$ dimensions. The marginal of a single component is the projection of
that uniform sphere onto one axis — uniform for $n=3$ (Archimedes), and **Gaussian** as $n$ grows,

```{math}
:label: eq-ideal-maxwell
p(v_x)=\sqrt{\frac{m}{2\pi kT}}\,e^{-mv_x^2/2kT},\qquad \operatorname{Var}(v_x)=\frac{R^2}{3N}=\frac{kT}{m},
```

the **Maxwell–Boltzmann** law (Maxwell 1860, the first statistical law in physics). The speed obeys
$f(v)\propto v^2 e^{-mv^2/2kT}$ with $\langle v\rangle=\sqrt{8kT/\pi m}$ and
$v_{\rm rms}=\sqrt{3kT/m}$. This is the geometric origin of the Boltzmann factor: the shadow of a
high-dimensional sphere (the [§5.3](large-n-limit.ipynb) Gaussian again).

### The Gibbs paradox: where classical counting fails

Without the $1/N!$, the entropy is not extensive: combining two boxes of the *same* gas predicts a
spurious increase,

```{math}
:label: eq-gibbs
\Delta S_{\rm spurious}=2Nk\ln 2\quad(\text{no }1/N!),\qquad \Delta S_{\rm same}\approx 0\quad(\text{with }1/N!).
```

The $1/N!$ restores extensivity. But indistinguishability, and the $h$ that fixes the *absolute*
entropy (Sackur–Tetrode), are **quantum** in origin: classical mechanics gives the form of the
entropy and every difference, but cannot fix its absolute scale or justify the $1/N!$ on its own.
We defer that to Volume VII — classical mechanics taken exactly as far as it honestly goes. The
entropy of mixing of *different* gases, by contrast, is real: $\Delta S=2Nk\ln 2$, the
$-k\sum x\ln x$ of [§5.3](large-n-limit.ipynb).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, Rectangle
from scipy.special import gammaln

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Units: m = h = k_B = 1 throughout (stated wherever it matters).
M, H, KB = 1.0, 1.0, 1.0


def ln_Sigma(E, V, N, gibbs=True):
    """The log phase-space volume ln Σ(E, V, N) of an ideal gas (eq-phase-volume).

    The volume of the accessible region with energy ≤ E: V^N (positions) times the
    3N-dimensional momentum ball of radius √(2mE), divided by h^(3N) N!. Computed in log
    space with ``scipy.special.gammaln`` (the Γ that regulates the n-ball volume), because
    Σ itself overflows any float for macroscopic N — the §5.3 / §0.1 lesson, now in
    3N dimensions. Units m = h = 1.

    Parameters
    ----------
    E, V : float
        Total energy and box volume.
    N : int
        Particle number.
    gibbs : bool, optional
        Include the Gibbs indistinguishability factor 1/N! (default ``True``). Setting it
        ``False`` exposes the Gibbs paradox (Exercise 8).

    Returns
    -------
    float
        The log phase-space volume ln Σ.
    """
    ln = (
        N * np.log(V)
        + 1.5 * N * np.log(2 * np.pi * M * E)
        - gammaln(1.5 * N + 1)
        - 3 * N * np.log(H)
    )
    if gibbs:
        ln -= gammaln(N + 1)
    return ln


def ln_g(E, V, N, gibbs=True):
    """The log density of states ln g(E) = ln(∂Σ/∂E) — the energy shell.

    Since Σ ∝ E^(3N/2), the density of states is g = ∂Σ/∂E = (3N/2E)·Σ, so
    ln g = ln Σ + ln(3N/2E). The shell of thickness ΔE has volume g·ΔE; concentration of
    measure (Exercise 2) makes its entropy agree with the ball's to O(ln N).

    Parameters
    ----------
    E, V : float
        Energy and volume.
    N : int
        Particle number.
    gibbs : bool, optional
        Pass through to :func:`ln_Sigma` (default ``True``).

    Returns
    -------
    float
        The log density of states ln g.
    """
    return ln_Sigma(E, V, N, gibbs) + np.log(1.5 * N / E)


def sample_velocity_sphere(N, E, m, n_samples, rng):
    """Sample velocities uniformly on the constant-energy sphere in 3N dimensions.

    For N free particles with fixed total kinetic energy E, the velocity vector lies on a sphere
    of radius R = √(2E/m) in 3N dimensions (the microcanonical measure). A uniform point on
    the sphere is a vector of independent Gaussians normalized to length R (``numpy.linalg.norm``).

    Parameters
    ----------
    N : int
        Particle number (3N velocity components).
    E, m : float
        Total kinetic energy and particle mass.
    n_samples : int
        Number of sphere points to draw.
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``).

    Returns
    -------
    numpy.ndarray, shape (n_samples, 3N)
        Velocity vectors, each of length √(2E/m).
    """
    R = np.sqrt(2 * E / m)
    g = rng.standard_normal((n_samples, 3 * N))
    g *= R / np.linalg.norm(g, axis=1, keepdims=True)
    return g


# The worked state: N particles, volume V, energy E (units m=h=k=1 → T=1, P=1).
N0, V0, E0 = 1000, 1000.0, 1500.0

## Exercise 1 — The phase-space volume of the ideal gas

We begin by building the multiplicity honestly, because everything rests on it. For a gas there is
nothing to count one-by-one: a microstate is a point in the $6N$-dimensional phase space of all
positions and momenta, and the microcanonical multiplicity is the **volume** of the accessible
region. That region factorizes ({numref}`fig-ig-box`): each particle's position ranges freely over
the box, giving $V^N$; and the momenta, constrained by $\sum p_i^2/2m\le E$, fill a **ball of
radius $\sqrt{2mE}$ in $3N$ dimensions**. The volume of an $n$-ball is $V_n(R)=\pi^{n/2}R^n/
\Gamma(n/2+1)$ — the $\Gamma$ function is what regulates "the volume of a ball" once $n$ is large
— so the whole phase-space volume is {eq}`eq-phase-volume`. Two prefactors appear that classical
mechanics cannot itself justify: $1/h^{3N}$, which makes $\Sigma$ dimensionless ($h$ is the size
of a phase-space cell), and the Gibbs factor $1/N!$ for indistinguishability. Both are quantum in
origin; we flag them now and resolve the $1/N!$ in Exercise 8. As [§5.3](large-n-limit.ipynb) warned, the volume itself
overflows instantly — in $3N=3000$ dimensions it has thousands of digits — so we compute
$\ln\Sigma$ with `scipy.special.gammaln`.

**This exercise (worked).** Assemble $\ln\Sigma$ from its three pieces ($N\ln V$, the $3N$-ball,
the $1/N!$) with `gammaln` and check it against the closed-form expression; anchor the
$\Gamma$-regulated ball formula to elementary geometry by checking the $N=1$ case against
$V\cdot\tfrac43\pi(2mE)^{3/2}$ computed directly; and confirm the raw volume overflows a float
while its logarithm stays finite.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    ln_sigma,
    ln_check,
    "the ideal-gas phase-space volume is the V^N × (3N-ball) expression",
    rtol=1e-12,
)
validate.close(
    ln_sigma_N1,
    ln_elementary,
    "the Γ-regulated ball formula reduces to the elementary (4/3)πR³ ball at N=1",
    rtol=1e-12,
)
validate.check(
    np.isinf(np.exp(ln_sigma)) and np.isfinite(ln_sigma),
    "the raw phase-space volume overflows a float; its logarithm via gammaln does not",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — Concentration of measure: shell versus ball

A worry must be dealt with before we differentiate, and dealing with it teaches a remarkable fact
about high-dimensional space. The microcanonical ensemble strictly lives on the energy *shell* —
energies in $[E,E+\Delta E]$ — not the filled ball of energies $\le E$. Does the entropy then
depend on the arbitrary thickness $\Delta E$, or on which we use? The resolution is
**concentration of measure**: in high dimensions a ball is almost entirely skin. The fraction of
an $n$-ball's volume outside radius $(1-\varepsilon)R$ is $1-(1-\varepsilon)^n$, which races to $1$
as $n$ grows ({numref}`fig-ig-concentration`) — for $n=300$, the outer *one percent* of the radius
already holds $95\%$ of the volume. So the ball is, for entropy purposes, indistinguishable from a
thin shell at its surface: $\ln\Sigma\approx\ln(g\,\Delta E)$ to $O(\ln N)$, and the temperature
we compute is the same either way. This is the same large-$N$ sharpening that made the equilibrium
peak razor-thin in [§5.3](large-n-limit.ipynb), now wearing its geometric hat.

**This exercise (worked).** Tabulate the outer-shell volume fraction $1-(1-\varepsilon)^n$ as $n$
climbs, and verify the $95\%$-at-$n{=}300$ fact by Monte Carlo: draw radii of uniform points in a
$300$-ball with `numpy.random.default_rng` (a uniform point's radius is distributed as $u^{1/n}$)
and measure the fraction beyond $0.99R$. Then compute the temperature from the **ball**
($\ln\Sigma$) and from the **shell** ($\ln g$) and confirm they agree to $O(1/N)$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    frac_300_mc,
    frac_300,
    "Monte Carlo confirms: 95% of a 300-ball's volume lies in the outer 1% of its radius",
    rtol=1e-2,
)
validate.close(
    T_ball,
    T_shell,
    "the shell and the ball give the same temperature to O(1/N) — the shell thickness does not matter",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Entropy and the caloric equation of state

Now the payoff begins. With the multiplicity in hand, entropy is $S=k\ln\Omega$ ([§5.4](microstates-entropy-temperature.ipynb)), and
temperature is the energy-slope $1/T=\partial S/\partial E$ — the *definition* we earned in [§5.4](microstates-entropy-temperature.ipynb),
applied now to the continuous gas. Because $\Sigma\propto E^{3N/2}$, that derivative is immediate:
$\partial(\ln\Sigma)/\partial E=3N/2E$, so $1/T=3N/2E$, which rearranges to the **caloric equation
of state** $E=\tfrac32 NkT$ {eq}`eq-eos`. This is **equipartition**, exact and from first
principles: each of the $3N$ translational degrees of freedom carries $\tfrac12 kT$ of energy. The
same statement we will keep meeting — every quadratic degree of freedom gets $\tfrac12 kT$ — here
falls straight out of the geometry of a high-dimensional ball.

**This exercise (worked).** Compute $1/T=\partial S/\partial E$ numerically (`numpy.gradient` over
a grid of energies), read off $T$ at the working energy, and confirm $E=\tfrac32 NkT$.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    E0,
    1.5 * N0 * T0,
    "1/T = ∂S/∂E gives the caloric equation of state E = (3/2)NkT (equipartition)",
    rtol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Pressure and the ideal gas law

The second derivative gives the second equation of state, and it is one of the most famous results
in physics — obtained here without ever assuming it. Pressure enters thermodynamics as the
volume-slope of entropy, $P/T=\partial S/\partial V$ {eq}`eq-eos`. From $\ln\Sigma$ the only
$V$-dependence is the positional factor $N\ln V$, so $\partial S/\partial V=N/V$, giving
$P/T=N/V$, that is $PV=NkT$ — the **ideal gas law** ({numref}`fig-ig-eos`, right). Boyle, Charles,
and Gay-Lussac measured this over centuries; here it drops out of the geometry of phase space in
one line. That is the power of the microcanonical method: the macroscopic equation of state is a
derivative of a counted (here, integrated) multiplicity.

**This exercise (worked).** Compute $P=T\,\partial S/\partial V$ (`numpy.gradient` over a grid of
volumes) and confirm $PV=NkT$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    P0 * V0,
    N0 * T0,
    "P/T = ∂S/∂V gives the thermal equation of state PV = NkT (the ideal gas law)",
    rtol=1e-3,
)

## Exercise 5 — The chemical potential

The third derivative names a quantity we have not met but will lean on heavily: the **chemical
potential** $\mu=-T\,\partial S/\partial N$ {eq}`eq-mu`, the change in energy when one particle is
added at fixed entropy and volume — equivalently, how much entropy a new particle brings. For a
dilute classical gas $\mu$ is *negative*: adding a particle at fixed total energy and volume raises
the entropy (there are more ways to arrange more particles), so $-T\,\partial S/\partial N<0$. The
chemical potential is the quantity that equalizes when particles are free to flow — across a
membrane, between phases, in a chemical reaction — exactly as temperature equalizes for energy
flow. It is the star of the grand canonical ensemble later in the volume, and the link from
statistical mechanics to chemistry.

**This exercise (worked).** Compute $\mu=-T\,\partial S/\partial N$ as a finite difference in the
integer $N$ (`ln_Sigma` at $N\pm1$), confirm it is negative for the dilute gas, and check it
against the closed form $\mu=-kT\ln\!\big[(V/N)(2\pi mkT)^{3/2}/h^3\big]$ that Stirling gives
for the ideal gas.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    mu,
    mu_closed,
    "μ = −T ∂S/∂N matches the closed form −kT ln[(V/N)(2πmkT)^(3/2)/h³]",
    rtol=1e-2,
)
validate.check(mu < 0, "the dilute classical gas has a negative chemical potential")

## Exercise 6 — The fundamental thermodynamic relation

Now we collect the three derivatives into a single statement that is, arguably, the master
equation of equilibrium thermodynamics. If $S$ is a function of $E$, $V$, and $N$, then any small
change in energy decomposes as $dU=T\,dS-P\,dV+\mu\,dN$ {eq}`eq-fundamental` — equivalently
$dS=\tfrac1T dU+\tfrac PT dV-\tfrac{\mu}{T}dN$, which is just the chain rule with our three slopes
$\partial S/\partial E=1/T$, $\partial S/\partial V=P/T$, $\partial S/\partial N=-\mu/T$. This one
differential packages the first law (energy bookkeeping) and the second (entropy as the state
function whose slopes are $T,P,\mu$). Everything in [§5.7](potentials-legendre-maxwell.ipynb) — the thermodynamic potentials, the
Legendre transforms that swap which variables are held fixed, the Maxwell relations — is built on
this relation. We can check it is no mere tautology by verifying it numerically: take a small,
simultaneous step in $(E,V,N)$ and confirm the *directly computed* entropy change matches the one
the relation predicts from the three slopes.

**This exercise (worked).** For a small step $(dE,dV,dN)$, compare $dS$ computed directly from
$\ln\Sigma$ to $\tfrac1T dU+\tfrac PT dV-\tfrac{\mu}{T}dN$ assembled from the three derivatives,
and confirm they agree.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    dS_direct,
    dS_relation,
    "the fundamental relation dU = TdS − PdV + μdN holds for the ideal gas",
    rtol=2e-2,
)

## Exercise 7 — Maxwell–Boltzmann as the shadow of the sphere

We turn from thermodynamics to the velocities themselves, and find the first statistical law ever
written in physics hiding in the geometry of a sphere. Fix the total kinetic energy: then the $3N$
velocity components satisfy $\sum\tfrac12 m v_i^2=E$, so the velocity vector lives *uniformly on a
sphere* of radius $R=\sqrt{2E/m}$ in $3N$ dimensions (the microcanonical measure for free
particles). What does one component look like? It is the **shadow** of that uniform sphere cast on
a single axis. For $n=3$ this shadow is exactly *uniform* (Archimedes' hat-box theorem); but as $n$
grows the shadow becomes a **Gaussian** ({numref}`fig-ig-projection`), by the same central-limit
mechanism as [§5.3](large-n-limit.ipynb) — the variance of one coordinate of a uniform $n$-sphere point is $R^2/n=kT/m$
{eq}`eq-ideal-maxwell`. That Gaussian is the **Maxwell–Boltzmann** velocity distribution (Maxwell, 1860),
and this is its deepest explanation: the famous factor $e^{-mv^2/2kT}$ is the *projection of a
high-dimensional energy sphere*. The speed of one particle then follows $f(v)\propto v^2
e^{-mv^2/2kT}$, with $\langle v\rangle=\sqrt{8kT/\pi m}$ and $v_{\rm rms}=\sqrt{3kT/m}$
({numref}`fig-ig-maxwell`).

**This exercise (worked).** Sample velocities uniformly on the $3N$-sphere (`sample_velocity_sphere`,
i.e. normalized Gaussians via `numpy.linalg.norm`); confirm one component is Gaussian with variance
$kT/m$; form one particle's speed and confirm the Maxwell–Boltzmann $\langle v\rangle$ and
$v_{\rm rms}$. The animation shows the single-component shadow morphing from uniform ($n=3$) to
Gaussian as the dimension grows — concentration of measure building the Boltzmann factor before our
eyes.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    v_component.var(),
    kT / M,
    "one velocity component is Gaussian with variance kT/m — Maxwell–Boltzmann as the projection of the energy sphere",
    rtol=3e-2,
)
validate.close(
    [v_mean, v_rms],
    [np.sqrt(8 * kT / (np.pi * M)), np.sqrt(3 * kT / M)],
    "the speed obeys the Maxwell–Boltzmann law",
    rtol=3e-2,
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The Gibbs paradox and the limit of classical counting

Here, at last, classical mechanics genuinely fails — and the *way* it fails is one of the most
instructive moments in the subject. Take two boxes of the *same* gas, each $(N,V,E)$, and remove
the wall. Physically nothing happens: it is the same gas at the same temperature and pressure, so
the entropy should not change. But if we use the naive multiplicity *without* the $1/N!$, the
computed entropy *jumps* by a spurious $2Nk\ln 2$ {eq}`eq-gibbs` — the **Gibbs paradox**
({numref}`fig-ig-gibbs`). Including the $1/N!$, which treats the particles as indistinguishable,
restores extensivity and the jump vanishes (up to an $O(\ln N)$ finite-size remnant that the
thermodynamic limit removes). Yet here is the honest catch: *why* $1/N!$, and the value of $h$ that
fixes the **absolute** entropy (the Sackur–Tetrode equation $S=Nk[\ln(V/N\lambda^3)+\tfrac52]$ with
$\lambda=h/\sqrt{2\pi mkT}$), are both **quantum** in origin. Classical mechanics gives the *form*
of the entropy and every *difference* correctly, but cannot justify the indistinguishability factor
or set the absolute scale on its own. We state Sackur–Tetrode, attribute its $\hbar$ and $1/N!$ to
quantum mechanics, and defer the derivation to Volume VII — classical physics taken exactly as far
as it honestly goes, the same move as Mercury's perihelion. And to be clear the paradox is about
*identical* gases: mixing two *different* gases really does raise the entropy by $2Nk\ln 2$, the
physical entropy of mixing $-k\sum x\ln x$ of [§5.3](large-n-limit.ipynb).

**This exercise (worked).** Combine two identical $(N,V,E)$ boxes into one $(2N,2V,2E)$: show the
entropy change is the spurious $2Nk\ln2$ *without* the Gibbs factor and $\approx0$ (per particle)
*with* it; then show the *different*-gas mixing entropy is the real $2Nk\ln2$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    dS_no_gibbs,
    2 * Ng * np.log(2),
    "without the Gibbs 1/N! the entropy gains a spurious 2Nk ln2 on combining identical gases (the Gibbs paradox)",
    rtol=2e-2,
)
validate.close(
    dS_gibbs / Ng,
    0.0,
    "the Gibbs 1/N! restores extensivity — combining identical gases changes the entropy per particle by ≈0",
    atol=2e-2,
)
validate.close(
    dS_mixing_different,
    2 * Ng * np.log(2),
    "mixing DIFFERENT gases gives the real entropy of mixing 2Nk ln2",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — Equipartition beyond translation, and the route to real gases

Before the synthesis, one extension that both broadens equipartition and plants a flag for what
classical mechanics cannot do. Our $E=\tfrac32 NkT$ counted $3N$ *translational* degrees of
freedom, each quadratic in a momentum, each carrying $\tfrac12 kT$. But equipartition counts *any*
quadratic degree of freedom — and a diatomic molecule can also *rotate*, adding two rotational
quadratic terms. So a classical diatomic gas has $5$ quadratic degrees of freedom per molecule and
a heat capacity $C_V=\tfrac52 Nk$ {eq}`eq-eos` (and a rigid polyatomic, $3$ rotations, gives $3Nk$).
This is the classical prediction — and it is *wrong* at low temperature, where measured heat
capacities fall below it as the rotational and vibrational modes "freeze out." That freezing is
quantum: the modes have discrete energy gaps that $kT$ cannot bridge when it is small, another
question deferred to Volume VII. (And for *interacting* gases the position integral no longer
factorizes into $V^N$ — the configurational integral with $U(r)$ is the route to the virial
expansion and the van der Waals equation, the foundational next step beyond the ideal gas.)

**This exercise (student).** First *measure* the translational heat capacity from the Exercise 3
temperature grid — $C_V=dE/dT$ with `numpy.gradient`, which must come out $\tfrac32Nk$ — then add
the $2$ rotational quadratic degrees of freedom of a diatomic molecule and confirm
$C_V=\tfrac52 Nk$, flagging that the low-temperature freeze-out of these modes is quantum.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    C_V_translational,
    1.5 * N0,
    "the measured microcanonical heat capacity is (3/2)Nk — ½k per quadratic degree of freedom",
    rtol=1e-2,
)
validate.close(
    C_V_diatomic,
    2.5 * N0,
    "adding the two rotational degrees of freedom gives the diatomic C_V = (5/2)Nk",
    rtol=1e-12,
)

## Exercise 10 — All of thermodynamics from a phase-space volume

Look back at what one integral bought. From the **volume of the energy shell** — an honest ball in
$3N$ dimensions, regulated by a $\Gamma$ function and computed in log space because it overflows —
we obtained the entropy $S=k\ln\Sigma$. Its three derivatives gave the **entire** equilibrium
thermodynamics of the gas: the caloric equation of state $E=\tfrac32NkT$, the thermal one
$PV=NkT$, the chemical potential $\mu=-T\,\partial S/\partial N$, and the **fundamental relation**
$dU=T\,dS-P\,dV+\mu\,dN$ that packages them. Concentration of measure assured us the shell and the
ball agree; the **Maxwell–Boltzmann** velocity law turned out to be the shadow of the energy sphere,
a Gaussian by the central-limit mechanism of [§5.3](large-n-limit.ipynb); and the one genuine failure — the absolute
entropy and the Gibbs $1/N!$ — became an honest deferral to the quantum volume. We bore the
full cost of doing statistical mechanics classically: a $6N$-dimensional integral rather than a
binomial coefficient. The next notebook ([§5.7](potentials-legendre-maxwell.ipynb)) takes the fundamental relation and builds the
thermodynamic potentials and Maxwell relations on it, with the Legendre transform of Volume II as
the key.

**This exercise (synthesis).** No new computation: the argument is the result. We compute one
high-dimensional volume and differentiate it three ways, and out comes an entire science. That a
macroscopic equation of state is a derivative of a microscopic phase-space volume is the whole idea
of statistical mechanics, and the ideal gas is where it is seen most cleanly.

## Notebook summary

From the phase-space volume of a classical gas, all of equilibrium thermodynamics followed by
differentiation — the heavier classical machinery developed in full, because the integral *is* the
physics.

- **The phase-space volume** {eq}`eq-phase-volume`: $\Sigma=V^N\pi^{3N/2}(2mE)^{3N/2}/(\Gamma(\tfrac{3N}{2}+1)h^{3N}N!)$,
  computed as $\ln\Sigma$ with `scipy.special.gammaln` because it overflows ([§5.3](large-n-limit.ipynb)/[§0.1](../00-foundations/floating-point.ipynb), in $3N$ dim).
- **Concentration of measure** {eq}`eq-concentration`: $1-(1-\varepsilon)^n\to1$ (the outer $1\%$
  of a $300$-ball holds $95\%$); the shell and ball give the same temperature to $O(1/N)$.
- **The equations of state** {eq}`eq-eos`: $1/T=\partial S/\partial E\Rightarrow E=\tfrac32NkT$
  (equipartition) and $P/T=\partial S/\partial V\Rightarrow PV=NkT$ (the ideal gas law, derived).
- **The chemical potential** {eq}`eq-mu`: $\mu=-T\,\partial S/\partial N\approx-2.76$ (negative for
  the dilute gas), the quantity that governs particle exchange.
- **The fundamental relation** {eq}`eq-fundamental`: $dU=T\,dS-P\,dV+\mu\,dN$, verified as a
  numerical identity — the master equation [§5.7](potentials-legendre-maxwell.ipynb) builds the potentials on.
- **Maxwell–Boltzmann** {eq}`eq-ideal-maxwell`: one velocity component is the Gaussian shadow of the
  energy sphere ($\operatorname{Var}=kT/m$), uniform at $n=3$ and Gaussian as $n$ grows; the speed
  law gives $\langle v\rangle=\sqrt{8kT/\pi m}$, $v_{\rm rms}=\sqrt{3kT/m}$.
- **The Gibbs paradox** {eq}`eq-gibbs`: $1/N!$ restores extensivity (spurious $2Nk\ln2\to0$); its
  quantum origin, and the absolute Sackur–Tetrode entropy, are deferred to Volume VII. Mixing
  *different* gases gives the real $2Nk\ln2$.

We bought the whole of equilibrium thermodynamics with one $6N$-dimensional integral.

## Outlook

- **Thermodynamic potentials and Maxwell relations ([§5.7](potentials-legendre-maxwell.ipynb)).** Take the fundamental relation and
  build the free energies by Legendre transform (the Volume II transform again), with the Maxwell
  relations as the integrability conditions — the working machinery of thermodynamics.
- **The chemical potential and the grand canonical ensemble.** $\mu$ comes into its own when
  particle number can fluctuate — the ensemble for open systems, later in the volume.
- **Real gases.** The configurational integral with interactions $U(r)$ no longer factorizes into
  $V^N$: the route to the virial expansion and the van der Waals equation, the foundational step
  beyond the ideal gas.
- **Quantum statistics (Volume VII).** The Sackur–Tetrode entropy *derived*, the $1/N!$ and $h$
  justified by indistinguishability, and the quantum gases (Fermi–Dirac, Bose–Einstein) where the
  freeze-out of internal modes and degeneracy take over.
- **Cross-reference** [§5.4](microstates-entropy-temperature.ipynb) (microcanonical entropy and $1/T=\partial S/\partial E$), [§5.3](large-n-limit.ipynb) (large-$N$,
  the Gaussian/CLT, the entropy of mixing), [§0.1](../00-foundations/floating-point.ipynb) (log space).

In [ ]:
from ecp.style import footer

footer()